GOAL- How does chunk_size and chunk_overlap affect retrieval quality?

Original notebook uses LlamaIndex + OpenAI for automated evaluation.
This local version uses LangChain + Hugging Face embeddings + FAISS + Ollama.
Goal: manually compare how chunk_size and chunk_overlap affect retrieval quality.

In [13]:
from pathlib import Path
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

import ollama

In [25]:
path = Path("../data/Understanding_Climate_Change.pdf")

loader = PyPDFLoader(str(path))
print(type(loader))
att = dir(loader)
# now I can filter out the dunder methods and attributes here
usable_func = [name for name in att if not name.startswith("__")]
print(usable_func)

# print(loader.__dict__)
print(help(loader.load))


<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>
['_abc_impl', '_is_s3_presigned_url', '_is_s3_url', '_is_valid_url', 'alazy_load', 'aload', 'file_path', 'headers', 'lazy_load', 'load', 'load_and_split', 'parser', 'source', 'web_path']
Help on method load in module langchain_core.document_loaders.base:

load() -> 'list[Document]' method of langchain_community.document_loaders.pdf.PyPDFLoader instance
    Load data into Document objects.
    
    Returns:
        the documents.

None


This is a debugging process here ::::
1) example: we will take the loader here-- type(loader) - we get to know what class it belongs to
2) to know all the attributes and methods -- dir(loader) 
3) in the above we can separate internal/dunder attributes and methods using a if not element.startswith(""__)
4) print(help(loader)) or print(help(loader.load)) --- one of the most useful and below is a implementation to know what are the attributes and methods of a class object that is has access to

In [22]:
public_names = [name for name in dir(loader) if not name.startswith("_")]

attributes = []
methods = []

for name in public_names:
    value = getattr(loader, name)
    if callable(value):   # if a value is callable means it is a method
        methods.append(name)
    else:
        attributes.append(name)

print("Attributes:")
print(attributes)

print("\nMethods:")
print(methods)

Attributes:
['file_path', 'headers', 'parser', 'source', 'web_path']

Methods:
['alazy_load', 'aload', 'lazy_load', 'load', 'load_and_split']


In [34]:
documents = loader.load()
print(type(documents))
print(dir(documents))
# print(help(documents))
# let's eliminate the dundar methods and attributes here and only show the usable external attributes and methods

doc = documents[0]
print(type(doc)) # found out that the class here is Document whose instance is this Doc
print([ele for ele in dir(doc) if not ele.startswith("__")])
print(help(doc.model_dump))
print(help(doc.model_dump_json))

<class 'list'>
['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']
<class 'langchain_core.documents.base.Document'>
['_abc_impl', '_calculate_keys', '_copy_and_set_values', '_get_value', '_iter', '_setattr_handler', 'construct', 'copy', 'dict', 'from_orm', 'get_lc_namespace', 'id', 'is_lc_serializable', 'json', 'lc_attributes', 'lc_id', 'lc_secrets', 'metadata', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_jso

In [35]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
def build_vector_store(chunk_size, chunk_overlap):
    # we initialized the text_splitter or we can say a chunker
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    vector_store = FAISS.from_documents(chunks, embedding_model)
    return vector_store, chunks

    
vector_store_1000, chunks_1000 = build_vector_store(
    chunk_size=1000,
    chunk_overlap=200,
)

print("Number of chunks:", len(chunks_1000))
print("First chunk preview:")
print(chunks_1000[0].page_content[:500])
print("First chunk metadata:")
print(chunks_1000[0].metadata)


Number of chunks: 97
First chunk preview:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. 
Historical Context 
The Earth's c
First chunk metadata:
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '../data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}


Now the bottleneck that we are studying is retrieval quality--> in RAG if the retrival brings bad chunks then LLM gets bad context, then even a good model may answer poorly

Small chunks
→ precise, but may miss surrounding context

Large chunks
→ more context, but may include noise

Overlap
→ helps when important info sits at the boundary between chunks

In [38]:
test_questions = [
    "What is the main cause of climate change?",
    "What are some effects of climate change?",
    "How does renewable energy help with climate change?",
    "What is ocean acidification?",
    "What does the document say about climate policy?",
]

In [ ]:
# here is the augmentation and generation 

def inspect_retrieval(vector_store, question, k=3):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    print("Question: ", question)
    print("Retrieved Chunks: ", len(retrieved_docs))

    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\n--- Retrieved chunk {i} ---")
        print("Page:", doc.metadata.get("page_label") or doc.metadata.get("page"))
        print(doc.page_content[:500])

    return retrieved_docs

In [42]:
from email import message
from urllib import response
import ollama

def generate_response(context, question):
    prompt = f""" 
        Answer the question with the help of the provided context:
        Question: {question}
        context: {context}
        Answer should be short and crisp.
    """
    response = ollama.chat(
        model="gemma4:e4B",
        messages=[
            {
                "role": "user",
                "content": prompt 
            }
        ]
    )
    print(response)
    return response.message.content

In [44]:
for que in test_questions:
    data = inspect_retrieval(vector_store_1000, que, k=1)
    print("Answer:::::",generate_response(data, que))

Question:  What is the main cause of climate change?
Retrieved Chunks:  1

--- Retrieved chunk 1 ---
Page: 1
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil F
model='gemma4:e4B' created_at='2026-07-30T07:23:43.418159Z' done=True done_reason='stop' total_duration=13320604583 load_duration=324089625 prompt_eval_count=412 prompt_eval_duration=142024000 eval_count=437 eval_duration=12842711000 message=Message(role='assistant', content='The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere, which has been 